In [ ]:
import tifffile
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

### This will use about 50GB of RAM, make sure you have enough free memory!

Files will be loaded from ZSeries... and saved in as TSeries...

Each file will be named TSeries...Slice0001....tif. (Contrary to ZSeries...Cycle0001...tif)

Make sure to change the top_dir

In [ ]:
# This assumes the only folders in top_dir are the folders of z slices
top_dir = "/home/ben/data/two-photon/test"
folders = os.listdir(top_dir)
for f in folders:
    # Check f directory is not empty
    if len(os.listdir(os.path.join(top_dir, f))) == 0:
        print(f"Skipping empty directory {f}")
        continue
    # Check f starts with Z (isn't already a TSeries)
    if not f.startswith("Z"):
        print(f"Skipping non-ZSeries directory {f}")
        continue
    print(f"Processing folder {f}")
    z_stacks = Path(os.path.join(top_dir, f))
    time_points = z_stacks.glob("*.tif")
    time_points = sorted([t for t in time_points if t.is_file()])
    print(f"Number of time points: {len(time_points)}")
    chunk = time_points
    volumes = [tifffile.imread(top_dir / c) for c in chunk]
    twod_plus_time = [np.stack([v[z] for v in volumes]) for z in range(6)]
    for t, vol in enumerate(twod_plus_time):
        new_folder_name = "T" + f[1:]  # Change Z to T
        file_name = f"{new_folder_name}_Slice{t+1:04d}_Ch2_000001.ome.tif"
        save_path = os.path.join(top_dir, new_folder_name, file_name)
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        print(f"Saving to {save_path}")
        tifffile.imwrite(save_path, vol)
    del volumes
    del twod_plus_time

The final tifs should have shape [T, H, W], and there should be 5 or 6 of them (for 5 or 6 z slices) per folder